# CSI5341 DEIM: Colab runner
Workflow: edit locally, `git push`, then re-run the **Train** cell here (it pulls the latest commit first).
First time in a session: run cells 1 and 2. Runtime type must be **GPU (T4)**.

One run = one arm + one seed. Run order: cell 3 (train), then cell 4 (test evaluation) when training has finished. Cell 5 aggregates every finished run.

In [ ]:
# 1. Settings + Drive
BRANCH = "CSI5341-Project"
DRIVE_ROOT = "/content/drive/MyDrive/CSI5341-DEIM"

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2. Setup: clone/pull code, install deps, restore data and weights (re-runnable, fast after the first time)
!mkdir -p /content && cd /content && ( [ -d CSI5341-DEIM ] || git clone -q -b {BRANCH} https://github.com/AliAdenAwil/CSI5341-DEIM.git )
!bash /content/CSI5341-DEIM/tools/colab_setup.sh {BRANCH} {DRIVE_ROOT}
%cd /content/CSI5341-DEIM

In [ ]:
# 3. Train one run. Pulls the latest pushed code first.
#    Resumes automatically if this run already has last.pth on Drive. Resuming restarts the random
#    stream, so for a MAL/VFL pair either finish both runs in one go or resume both at the same epoch.
ARM   = "mal_dense"    # mal_dense | vfl_dense | mal_plain | vfl_plain | ref_imagenet
SEED  = 0
EXTRA = ""             # extra -u overrides for the pilot only, e.g. "epoches=10"
RUN_NAME = f"{ARM}_s{SEED}"     # prefix with e.g. "pilot_" for runs that must not enter the analysis

import os, shlex
!bash tools/colab_setup.sh {BRANCH} {DRIVE_ROOT} > /dev/null && git log -1 --format='code: %h %s'

CONFIG = f"configs/pets/{ARM}.yml"
out = f"{DRIVE_ROOT}/outputs/{RUN_NAME}"
last = f"{out}/last.pth"
if os.path.exists(last):
    init = f"-r {shlex.quote(last)}"
    print("resuming from", last)
elif ARM != "ref_imagenet":
    init = "-t weights/deim_dfine_hgnetv2_n_coco_160e.pth"
else:
    init = ""          # reference run: ImageNet backbone only
overrides = f"HGNetv2.local_model_dir={DRIVE_ROOT}/weights/hgnetv2/ {EXTRA}"
!python train.py -c {CONFIG} {init} --seed {SEED} --use-amp --output-dir {shlex.quote(out)} -u {overrides}

In [ ]:
# 4. Test evaluation of the run from cell 3: the checkpoint with the best val AP, evaluated once on the test split.
#    Writes eval_test.json into the run folder. Not needed for ref_imagenet.
ckpt = next(p for p in (f"{out}/best_stg2.pth", f"{out}/best_stg1.pth") if os.path.exists(p))
print("evaluating", ckpt)
!python train.py -c {CONFIG} -r {shlex.quote(ckpt)} --test-only --output-dir {shlex.quote(out)} -u val_dataloader.dataset.ann_file=./data/oxford_pets/coco/test.json | tail -n 16

In [ ]:
# 4b. Run a queue of runs back to back (train, then test evaluation). Use this instead of cells 3 and 4.
#     Finished runs are skipped, so the cell is safe to re-run after a disconnect.
QUEUE = [
    ("vfl_dense", 0),
    ("ref_imagenet", 0),
    ("mal_dense", 1),
    # ("vfl_dense", 1), ("mal_dense", 2), ("vfl_dense", 2), ("mal_plain", 0), ("vfl_plain", 0),
]

import os, shlex, json
get_ipython().system(f"bash tools/colab_setup.sh {BRANCH} {DRIVE_ROOT} > /dev/null && git log -1 --format='code: %h %s'")
SHOW = "grep --line-buffered -E 'best_stat|Training time|Traceback|Error|resum|Saved eval'"

def epochs_done(out):
    # log.txt has one JSON line per completed epoch; the last line's epoch is the truth even after a resume
    p = f"{out}/log.txt"
    if not os.path.exists(p):
        return 0
    lines = [l for l in open(p) if l.strip()]
    return json.loads(lines[-1])["epoch"] + 1 if lines else 0

for ARM, SEED in QUEUE:
    CONFIG = f"configs/pets/{ARM}.yml"
    out = f"{DRIVE_ROOT}/outputs/{ARM}_s{SEED}"
    total = 29 if ARM == "ref_imagenet" else 58
    print(f"\n===== {ARM}_s{SEED}: {epochs_done(out)}/{total} epochs done =====")

    if epochs_done(out) < total:
        last = f"{out}/last.pth"
        if os.path.exists(last):
            init = f"-r {shlex.quote(last)}"
            print("WARNING: resuming a partial run; its MAL/VFL partner no longer shares the same data order")
        elif ARM != "ref_imagenet":
            init = "-t weights/deim_dfine_hgnetv2_n_coco_160e.pth"
        else:
            init = ""
        os.makedirs(out, exist_ok=True)
        get_ipython().system(
            f"python train.py -c {CONFIG} {init} --seed {SEED} --use-amp --output-dir {shlex.quote(out)} "
            f"-u HGNetv2.local_model_dir={DRIVE_ROOT}/weights/hgnetv2/ 2>&1 | tee -a {shlex.quote(out)}/stdout.log | {SHOW}")
        if epochs_done(out) < total:
            print(f"STOPPED: {ARM}_s{SEED} did not finish; see {out}/stdout.log")
            break

    if ARM != "ref_imagenet" and not os.path.exists(f"{out}/eval_test.json"):
        ckpt = next(p for p in (f"{out}/best_stg2.pth", f"{out}/best_stg1.pth") if os.path.exists(p))
        get_ipython().system(
            f"python train.py -c {CONFIG} -r {shlex.quote(ckpt)} --test-only --output-dir {shlex.quote(out)} "
            f"-u val_dataloader.dataset.ann_file=./data/oxford_pets/coco/test.json 2>&1 | {SHOW}")
print("\nqueue finished")

In [ ]:
# 5. Aggregate all finished runs: tables, decision rule, figures (written to DRIVE_ROOT/outputs/analysis)
!python tools/analyze_runs.py --root {DRIVE_ROOT}/outputs
from IPython.display import Image, display
for f in ("low_iou_fraction.png", "val_ap.png"):
    p = f"{DRIVE_ROOT}/outputs/analysis/{f}"
    if os.path.exists(p): display(Image(p, width=600))